In [1]:
import os

import numpy as np
import skimage.io as io

from deepcell.utils.plot_utils import create_rgb_image
from deepcell.utils.plot_utils import make_outline_overlay

npz_dir = r"C:\Users\asus\.deepcell\datasets\tissuenet_v1-1"
train_dict = np.load(os.path.join(npz_dir, 'train.npz'),allow_pickle=True)
val_dict = np.load(os.path.join(npz_dir, 'val.npz'),allow_pickle=True)
test_dict = np.load(os.path.join(npz_dir, 'test.npz'),allow_pickle=True)


In [2]:
train_X, train_y, train_meta = train_dict['X'], train_dict['y'], train_dict['meta']
val_X, val_y, val_meta = val_dict['X'], val_dict['y'], val_dict['meta']
test_X, test_y, test_meta = test_dict['X'], test_dict['y'], test_dict['meta']

In [3]:
import pandas as pd

train_df = pd.DataFrame(train_meta[1:], columns=train_meta[0])

test_df = pd.DataFrame(test_meta[1:], columns=test_meta[0])
test_df = test_df.loc[3:,:].reset_index(drop=True)

val_df = pd.DataFrame(val_meta[1:], columns=val_meta[0])

In [4]:
train_specimens = set(train_df["specimen"])
val_specimens = set(val_df["specimen"])
test_specimens = set(test_df["specimen"])

print("train = val:", train_specimens == val_specimens)
print("train = test:", train_specimens == test_specimens)
print("val = test:", val_specimens == test_specimens)

train_df['specimen'].value_counts()

train = val: True
train = test: True
val = test: True


specimen
Pancreas                 606
Colon                    452
Breast                   379
Tonsil                   335
Esophagus                333
lymph node metastasis    295
Lymph Node                89
Epidermis                 86
Spleen                     4
Lung                       1
Name: count, dtype: int64

In [12]:
colon_train = (train_df['specimen'] == 'Colon').sum()
colon_val = (val_df['specimen'] == 'Colon').sum()
colon_test = (test_df['specimen'] == 'Colon').sum()

print(f'Number of Colon \nTrain: {colon_train}, Val: {colon_val}, Test: {colon_test}')

Number of Colon 
Train: 452, Val: 471, Test: 228


## RQ2: How much does segmentation performance degrade on unseen tissue types?

train: exclude colon  
val: exclude colon  
ID test: exclude colon  
OOD test: only colon  

In [51]:
held_out_tissue = "Colon"

train_mask = (train_df["specimen"] != held_out_tissue)
train_indices = train_df.index[train_mask].tolist()

In [39]:
import torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt
from IPython.display import clear_output
from torch.nn.functional import relu

In [40]:
class UNet(nn.Module):
    def __init__(self, n_class):
        super().__init__()

        # Encoder
        # input: 2x256x256
        self.e11 = nn.Conv2d(2, 64, kernel_size=3, padding=1)   # output: 64x256x256
        self.e12 = nn.Conv2d(64, 64, kernel_size=3, padding=1)  # output: 64x256x256
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)      # output: 64x128x128

        # input: 64x128x128
        self.e21 = nn.Conv2d(64, 128, kernel_size=3, padding=1)   # output: 128x128x128
        self.e22 = nn.Conv2d(128, 128, kernel_size=3, padding=1)  # output: 128x128x128
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)        # output: 128x64x64

        # input: 128x64x64
        self.e31 = nn.Conv2d(128, 256, kernel_size=3, padding=1)  # output: 256x64x64
        self.e32 = nn.Conv2d(256, 256, kernel_size=3, padding=1)  # output: 256x64x64
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)        # output: 256x32x32

        # input: 256x32x32
        self.e41 = nn.Conv2d(256, 512, kernel_size=3, padding=1)  # output: 512x32x32
        self.e42 = nn.Conv2d(512, 512, kernel_size=3, padding=1)  # output: 512x32x32
        self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2)        # output: 512x16x16

        # input: 512x16x16
        self.e51 = nn.Conv2d(512, 1024, kernel_size=3, padding=1)   # output: 1024x16x16
        self.e52 = nn.Conv2d(1024, 1024, kernel_size=3, padding=1)  # output: 1024x16x16

        # Decoder
        self.upconv1 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)  # output: 512x32x32
        # concatenated with e42: 1024x32x32
        self.d11 = nn.Conv2d(1024, 512, kernel_size=3, padding=1)  # output: 512x32x32
        self.d12 = nn.Conv2d(512, 512, kernel_size=3, padding=1)   # output: 512x32x32

        self.upconv2 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)  # output: 256x64x64
        # concatenated with e32: 512x64x64
        self.d21 = nn.Conv2d(512, 256, kernel_size=3, padding=1)  # output: 256x64x64
        self.d22 = nn.Conv2d(256, 256, kernel_size=3, padding=1)  # output: 256x64x64

        self.upconv3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)  # output: 128x128x128
        # concatenated with e22: 256x128x128
        self.d31 = nn.Conv2d(256, 128, kernel_size=3, padding=1)  # output: 128x128x128
        self.d32 = nn.Conv2d(128, 128, kernel_size=3, padding=1)  # output: 128x128x128

        self.upconv4 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)  # output: 64x256x256
        # concatenated with e12: 128x256x256
        self.d41 = nn.Conv2d(128, 64, kernel_size=3, padding=1)  # output: 64x256x256
        self.d42 = nn.Conv2d(64, 64, kernel_size=3, padding=1)   # output: 64x256x256

        # Output layer
        self.outconv = nn.Conv2d(64, n_class, kernel_size=1)  # output: n_classx256x256
        
    def forward(self, x):
        # Encoder
        xe11 = relu(self.e11(x))
        xe12 = relu(self.e12(xe11))
        xp1 = self.pool1(xe12)

        xe21 = relu(self.e21(xp1))
        xe22 = relu(self.e22(xe21))
        xp2 = self.pool2(xe22)

        xe31 = relu(self.e31(xp2))
        xe32 = relu(self.e32(xe31))
        xp3 = self.pool3(xe32)

        xe41 = relu(self.e41(xp3))
        xe42 = relu(self.e42(xe41))
        xp4 = self.pool4(xe42)

        xe51 = relu(self.e51(xp4))
        xe52 = relu(self.e52(xe51))
        
        # Decoder
        xu1 = self.upconv1(xe52)
        xu11 = torch.cat([xu1, xe42], dim=1)
        xd11 = relu(self.d11(xu11))
        xd12 = relu(self.d12(xd11))

        xu2 = self.upconv2(xd12)
        xu22 = torch.cat([xu2, xe32], dim=1)
        xd21 = relu(self.d21(xu22))
        xd22 = relu(self.d22(xd21))

        xu3 = self.upconv3(xd22)
        xu33 = torch.cat([xu3, xe22], dim=1)
        xd31 = relu(self.d31(xu33))
        xd32 = relu(self.d32(xd31))

        xu4 = self.upconv4(xd32)
        xu44 = torch.cat([xu4, xe12], dim=1)
        xd41 = relu(self.d41(xu44))
        xd42 = relu(self.d42(xd41))

        # Output layer
        out = self.outconv(xd42)

        return out

In [41]:
def random_crop(image, label, crop_size=256):
    height, width = image.shape[:2]

    top = np.random.randint(0, height - crop_size + 1)
    left = np.random.randint(0, width - crop_size + 1)

    image = image[top:top + crop_size, left:left + crop_size, :]
    label = label[top:top + crop_size, left:left + crop_size, :]

    return image, label

In [42]:
class TissueNetDataset(Dataset):
    def __init__(self, images, labels, training=False, crop_size=256):
        self.images = images
        self.labels = labels
        self.training = training
        self.crop_size = crop_size

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        image = self.images[index]
        label = self.labels[index]

        if self.training:
            image, label = random_crop(image, label, self.crop_size)

        # Normalize each channel to [0, 1].
        image = image.astype(np.float32, copy=True)

        low = np.percentile(image, 1, axis=(0, 1), keepdims=True).astype(np.float32)
        high = np.percentile(image, 99, axis=(0, 1), keepdims=True).astype(np.float32)

        scale = np.maximum(high - low, np.float32(1e-6))
        np.subtract(image, low, out=image)
        np.divide(image, scale, out=image)
        np.clip(image, 0, 1, out=image)

        # Instance IDs -> binary masks.
        label = (label > 0).astype(np.float32)

        # HWC -> CHW.
        image = torch.from_numpy(np.ascontiguousarray(image.transpose(2, 0, 1))).float()
        label = torch.from_numpy(np.ascontiguousarray(label.transpose(2, 0, 1))).float()

        return image, label

In [ ]:
from torch.utils.data import Subset

train_dataset_full = TissueNetDataset(train_X, train_y, training=True)
train_dataset = Subset(train_dataset_full, train_indices)

In [54]:
len(train_dataset_full), len(train_dataset)

(2580, 2128)

(tensor([[[0.0000, 0.0000, 0.0000,  ..., 1.0000, 1.0000, 0.5833],
          [0.0000, 0.0000, 0.0000,  ..., 0.6667, 0.5000, 0.8333],
          [0.0000, 0.0000, 0.0000,  ..., 0.2500, 0.1667, 0.5833],
          ...,
          [0.0000, 0.0833, 0.0833,  ..., 0.3333, 0.3333, 0.5833],
          [0.0000, 0.0000, 0.0000,  ..., 0.0833, 0.2500, 0.2500],
          [0.0000, 0.0833, 0.0000,  ..., 0.0833, 0.0000, 0.3333]],
 
         [[0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
          [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.6667, 0.0000],
          [0.0000, 0.0000, 0.0000,  ..., 0.6667, 0.0000, 0.0000],
          ...,
          [0.0000, 0.0000, 0.3333,  ..., 1.0000, 0.6667, 0.3333],
          [0.6667, 0.6667, 0.3333,  ..., 0.3333, 0.0000, 0.0000],
          [0.3333, 0.6667, 0.6667,  ..., 0.0000, 0.6667, 0.0000]]]),
 tensor([[[0., 0., 0.,  ..., 1., 1., 0.],
          [0., 0., 0.,  ..., 1., 1., 0.],
          [0., 0., 0.,  ..., 1., 1., 0.],
          ...,
          [1., 1., 1.,  ..., 0.